In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

### Notes
* Need to use logistic regression because this is a classification problem
* Vectorizer is TF-IDF: 
    - TF (Term Freq): How often term appears in ngram in a single report
    - IDF (Inverse Docuiment Freq): How rare ngram across all reports
    - High weight -> Bigger signal for Logistic Regression to use!
    - example: 
        * "the" appears alot, its weight is small (↑TF x ↓↓↓IDF = ↓↓), 
        * "acl" is rare so it has a high weight  (TF=↓ x ↑↑↑IDF = ↑↑)
    - [[0.    0.15  0.    0.32  0.    0.08  ...]]

In [17]:
train_df = pd.read_csv("train.csv")
sample_submission = pd.read_csv("sample_submission.csv")


In [18]:
train_df["Report"] = train_df["Report"].fillna("")
train_df

,StudyInstanceUID,Report,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10004873229099053869...,Técnica: RMN de la rodilla. Resultados: Rotura...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1.2.826.0.1.3680043.8.498.10004945927472656027...,[DATE]: * MR Knie Rechts 15ch AA Klinische Inl...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1.2.826.0.1.3680043.8.498.10009278692606631573...,Hallazgos:\nNo hay alteraciones en significati...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1.2.826.0.1.3680043.8.498.10009639203170750274...,"In the medial compartment, the meniscus is no...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1.2.826.0.1.3680043.8.498.10013663742400736029...,CONSTATATIONS :\n\nFractures :\nAucune.\n\nAli...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4402,1.2.826.0.1.3680043.8.498.99880021280666757817...,Exam Type: MRI KNEE RIGHT WO CONTRAST\nExam Da...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4403,1.2.826.0.1.3680043.8.498.99881620336512626242...,[DATE]: *MR Knie Rechts 15ch AA Klinische Inli...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4404,1.2.826.0.1.3680043.8.498.99926624968240681772...,"SAĞ DİZ MRG. Tetkik protokolü: Çok düzlemli, ç...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4405,1.2.826.0.1.3680043.8.498.99939100657291954890...,"MRI of Knee with \n-Locator, SG PD FatSat, SG ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
target_cols = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
]

X = train_df["Report"]
y = train_df[target_cols].astype(float)


(4407, 14)
Effusion            0.603448
Synovitis           0.465517
Medial Meniscus     0.448276
Medial Meniscus     0.448276
ACL                 0.413793
Lateral Meniscus    0.396552
Lateral Meniscus    0.396552
PF OA               0.362069
Contusion           0.327586
Fracture            0.310345
Medial OA           0.258621
Baker's             0.206897
Lateral OA          0.189655
MCL                 0.155172
dtype: float64


In [28]:
X

0       Técnica: RMN de la rodilla. Resultados: Rotura...
1       [DATE]: * MR Knie Rechts 15ch AA Klinische Inl...
2       Hallazgos:\nNo hay alteraciones en significati...
3        In the medial compartment, the meniscus is no...
4       CONSTATATIONS :\n\nFractures :\nAucune.\n\nAli...
                              ...                        
4402    Exam Type: MRI KNEE RIGHT WO CONTRAST\nExam Da...
4403    [DATE]: *MR Knie Rechts 15ch AA Klinische Inli...
4404    SAĞ DİZ MRG. Tetkik protokolü: Çok düzlemli, ç...
4405    MRI of Knee with \n-Locator, SG PD FatSat, SG ...
4406    Técnica: RMN de la rodilla. Resultados: Rotura...
Name: Report, Length: 4407, dtype: str

In [27]:
y

,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4402,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4403,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4404,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4405,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

In [21]:
vectorizer = TfidfVectorizer(
    analyzer="char", 
    ngram_range=(3, 5), 
    min_df=2, 
    max_features=10000
    )

X_train_vec = vectorizer.fit_transform(X_train)
X_valid_vec = vectorizer.transform(X_valid)


In [22]:
X_train_vec[:5, :20].toarray()

array([[0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.0061456 , 0.00620884,
        0.00620884, 0.        , 0.0052199 , 0.00604667, 0.00605203,
        0.        , 0.        , 0.        , 0.00555764, 0.00562837],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0

In [23]:
# Get the actual feature names (n-grams)
feature_names = vectorizer.get_feature_names_out()
print(feature_names[:20])

['\n1.' '\n2.' '\n2. ' '\n3.' '\n> ' '\nal' '\nali' '\nalig' '\nan'
 '\nant' '\nante' '\nca' '\nco' '\ncom' '\ncomp' '\ncon' '\ncond' '\nde'
 '\ndi' '\ndis']


In [24]:
model = OneVsRestClassifier(LogisticRegression(max_iter=1000, solver='liblinear'))